# 01 · Data import and quality control (QC)

Loads one data source into the common format (patients, visits, treatments, comorbidities), checks it against the schemas and clinical coherence rules, and applies the documented cleaning rules.

**Output:** QC report for the thesis (*Data quality assessment*) + clean tables in `data/interim/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import params, path
pd.set_option("display.max_columns", 50)

In [ ]:
SOURCE = "fictitious"   # or "kruczek_il17"

if SOURCE == "fictitious":
    from src.data.loaders.fictitious import load
elif SOURCE == "kruczek_il17":
    from src.data.loaders.kruczek_il17 import load

raw = load()
{k: v.shape for k, v in raw.items()}

## 1. Overview

In [ ]:
from src.data.qc import run_qc
qc = run_qc(raw)
qc["summary"]

## 2. Schema issues
Values out of the valid range (PASI 0–72, BSA 0–100, DLQI 0–30), invalid categories, wrong types.

In [ ]:
qc["schema_issues"]

## 3. Coherence issues
Problems a schema cannot see: duplicate visits, PASI/BSA contradictions, end before start, orphan records.

In [ ]:
qc["coherence_issues"]

In [ ]:
# Inspect the rows behind each problem
for _, row in qc["coherence_issues"].iterrows():
    print(row.problem)
    display(raw[row.table].loc[[row["index"]]])

## 4. Missing values

In [ ]:
miss = qc["missingness"]
miss[miss.n_missing > 0].sort_values("pct_missing", ascending=False)

## 5. Cleaning
Rules in `src/data/clean.py`. Every change is logged so it can be reported.

In [ ]:
from src.data.clean import clean
tables, cleaning_log = clean(raw)
cleaning_log

In [ ]:
# QC again after cleaning: schema and coherence issues should be gone
qc_after = run_qc(tables)
print("Schema issues:", len(qc_after["schema_issues"]))
print("Coherence issues:", len(qc_after["coherence_issues"]))

## 6. Save

In [ ]:
out = path("interim") / SOURCE
out.mkdir(parents=True, exist_ok=True)
for name, df in tables.items():
    df.to_csv(out / f"{name}.csv", index=False)

# QC report tables (for the thesis)
tab = path("tables")
qc["schema_issues"].to_csv(tab / f"qc_schema_issues_{SOURCE}.csv", index=False)
qc["coherence_issues"].to_csv(tab / f"qc_coherence_issues_{SOURCE}.csv", index=False)
qc["missingness"].to_csv(tab / f"qc_missingness_{SOURCE}.csv", index=False)
cleaning_log.to_csv(tab / f"qc_cleaning_log_{SOURCE}.csv", index=False)
print("Saved to", out)